<a href="https://colab.research.google.com/github/azharnoor864-spec/NLP_Pipeline/blob/main/NLP_Pipeline_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
!pip install PyMuPDF pdfplumber

In [31]:
import os
import re
import fitz  # PyMuPDF
import pdfplumber
import requests

# Define local storage directories
PDF_DIR = "downloaded_pdfs"
IMAGE_DIR = "extracted_images"
os.makedirs(PDF_DIR, exist_ok=True)
os.makedirs(IMAGE_DIR, exist_ok=True)

# Document sources with text, tables, and images from different domains
PDF_SOURCES = {
    "research_paper": "https://www.ijrti.org/papers/IJRTI2304061.pdf",
    "technical_manual": "https://stars.aashe.org/wp-content/uploads/2024/05/STARS-Technical-Manual-v3.0.pdf",
    "news_article": "https://gmmdc.mandela.ac.za/mbeki-maths-dev-n/media/GMMDC-Project-FIles/Main%20Page/Centre%20Docs/GMMDC-Media-Articles-2019.pdf?ext=.pdf"
}

def download_pdfs(sources: dict, output_dir: str) -> dict:
    """
    Downloads PDF files from web URLs and saves them locally.
    """
    local_paths = {}
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

    for label, url in sources.items():
        local_path = os.path.join(output_dir, f"{label}.pdf")
        print(f"📥 Downloading {label}...")
        try:
            response = requests.get(url, headers=headers, timeout=30)
            response.raise_for_status()
            with open(local_path, "wb") as f:
                f.write(response.content)
            local_paths[label] = local_path
            print(f"💾 Saved to {local_path}")
        except Exception as e:
            print(f"❌ Failed to download {label}: {e}")
    return local_paths

def is_valid_table(grid: list) -> bool:
    """
    Stricter table validator.
    Filters out text sidebars, banners, and layout blocks using content ratios.
    """
    if not grid or len(grid) < 2:
        return False

    # Standardize column count based on the first row
    expected_cols = len(grid[0])
    if expected_cols < 2:
        return False

    total_cells = 0
    filled_cells = 0
    long_text_cells = 0

    for row in grid:
        if not row:
            continue
        for cell in row:
            total_cells += 1
            if cell is not None and str(cell).strip() != "":
                filled_cells += 1
                # If a single cell contains more than 15 words, it's likely a regular text paragraph
                if len(str(cell).split()) > 15:
                    long_text_cells += 1

    if total_cells == 0:
        return False

    # Calculate metrics
    fill_rate = filled_cells / total_cells
    long_text_ratio = long_text_cells / filled_cells if filled_cells > 0 else 0

    # 1. Reject if the grid is mostly empty space (common with structural layout borders)
    if fill_rate < 0.25:
        return False

    # 2. Reject if cells contain long narrative sentences instead of tabular data values
    if long_text_ratio > 0.30:
        return False

    return True

def extract_pdf_content(file_path: str, source_label: str, img_dir: str) -> dict:
    """
    Extracts text, structural tables, and embedded images from a single PDF file.
    Filters out layout blocks using cell text density analysis.
    """
    extracted_data = {
        "source": source_label,
        "text": "",
        "tables": [],
        "image_count": 0
    }

    print(f"\n🔍 Extracting assets from: {source_label}...")

    # 1. Extract Text and Valid Tables using pdfplumber
    with pdfplumber.open(file_path) as pdf:
        raw_table_count = 0
        for page_num, page in enumerate(pdf.pages, start=1):
            # Extract Text
            page_text = page.extract_text()
            if page_text:
                extracted_data["text"] += f"\n--- Page {page_num} ---\n" + page_text

            # Extract and strictly filter tables
            tables = page.extract_tables()
            for table in tables:
                raw_table_count += 1
                if is_valid_table(table):
                    extracted_data["tables"].append({
                        "page": page_num,
                        "data": table
                    })

        print(f"📊 Table Filter: Reduced {raw_table_count} elements down to {len(extracted_data['tables'])} genuine tables.")

    # 2. Extract Embedded Images using PyMuPDF (fitz)
    try:
        doc = fitz.open(file_path)
        image_idx = 0
        for page_num in range(len(doc)):
            page = doc[page_num]
            image_list = page.get_images(full=True)

            for img in image_list:
                xref = img[0]
                base_image = doc.extract_image(xref)
                if not base_image:
                    continue

                image_bytes = base_image["image"]
                image_ext = base_image["ext"]

                image_name = f"{source_label}_pg{page_num+1}_img{image_idx}.{image_ext}"
                image_path = os.path.join(img_dir, image_name)

                with open(image_path, "wb") as f:
                    f.write(image_bytes)

                image_idx += 1

        extracted_data["image_count"] = image_idx
    except Exception as e:
        print(f"⚠️ Error during image extraction for {source_label}: {e}")

    print(f"✅ Finished {source_label}. Extracted {len(extracted_data['text'])} chars and {extracted_data['image_count']} images.")
    return extracted_data

# --- Execution Entry Point ---
if __name__ == "__main__":
    downloaded_files = download_pdfs(PDF_SOURCES, PDF_DIR)

    parsed_corpus = {}
    for label, path in downloaded_files.items():
        if os.path.exists(path):
            parsed_corpus[label] = extract_pdf_content(path, label, IMAGE_DIR)


📥 Downloading research_paper...
💾 Saved to downloaded_pdfs/research_paper.pdf
📥 Downloading technical_manual...
💾 Saved to downloaded_pdfs/technical_manual.pdf
📥 Downloading news_article...
💾 Saved to downloaded_pdfs/news_article.pdf

🔍 Extracting assets from: research_paper...
📊 Table Filter: Reduced 23 elements down to 0 genuine tables.
✅ Finished research_paper. Extracted 16432 chars and 6 images.

🔍 Extracting assets from: technical_manual...
📊 Table Filter: Reduced 279 elements down to 195 genuine tables.
✅ Finished technical_manual. Extracted 504441 chars and 32 images.

🔍 Extracting assets from: news_article...
📊 Table Filter: Reduced 1 elements down to 0 genuine tables.
✅ Finished news_article. Extracted 182 chars and 18 images.


In [32]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Automatically fetch required text resources from NLTK
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

def split_text_into_chunks(text: str, chunk_size: int = 500, overlap: int = 50) -> list:
    """
    Splits long strings of text into smaller overlapping chunks.

    Args:
        text (str): Raw extracted document text.
        chunk_size (int): Character length limit for each chunk.
        overlap (int): Number of overlapping characters between neighboring chunks.

    Returns:
        list: Extracted string chunks.
    """
    chunks = []
    start_idx = 0
    while start_idx < len(text):
        end_idx = start_idx + chunk_size
        chunk = text[start_idx:end_idx]
        chunks.append(chunk.strip())
        start_idx += (chunk_size - overlap)
    return chunks

def clean_and_lemmatize_text(text: str) -> str:
    """
    Normalizes a text chunk by lowercasing, removing special characters,
    filtering out English stopwords, and stripping suffixes to leave root words.

    Args:
        text (str): Raw text block string.

    Returns:
        str: Tokenized, cleaned, and lemmatized string.
    """
    # 1. Lowercase and remove all non-alphanumeric text characters
    text_lowercase = text.lower()
    clean_chars = re.sub(r'[^a-z0-9\s]', '', text_lowercase)
    words = clean_chars.split()

    # 2. Setup NLTK linguistic processing tools
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()

    # 3. Drop stopwords and convert words to their base roots (e.g., 'running' -> 'run')
    processed_tokens = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]

    return " ".join(processed_tokens)

def build_cleaned_corpus(parsed_corpus: dict) -> list:
    """
    Takes the raw parsed PDFs, chunks their text, applies advanced cleaning,
    and returns a clean dataset ready for mathematical vector matching.

    Args:
        parsed_corpus (dict): Output dictionary from the Phase 1 parsing script.

    Returns:
        list: Collection of structured chunk dictionaries.
    """
    final_processed_chunks = []

    for source_label, doc_data in parsed_corpus.items():
        print(f"🧹 Chunking and cleaning text vectors for domain: {source_label}...")

        # Split document narrative text into clean fragments
        raw_chunks = split_text_into_chunks(doc_data["text"])

        for idx, chunk in enumerate(raw_chunks):
            # Run our advanced cleaning pipeline
            cleaned_chunk = clean_and_lemmatize_text(chunk)

            # Save both variants: raw text for the LLM context, cleaned text for mathematical search
            final_processed_chunks.append({
                "chunk_id": f"{source_label}_{idx}",
                "source": source_label,
                "raw_text": chunk,
                "cleaned_text": cleaned_chunk
            })

    print(f"✨ Success! Generated {len(final_processed_chunks)} clean searchable vectors across all documents.")
    return final_processed_chunks
chunked_corpus = build_cleaned_corpus(parsed_corpus)

🧹 Chunking and cleaning text vectors for domain: research_paper...
🧹 Chunking and cleaning text vectors for domain: technical_manual...
🧹 Chunking and cleaning text vectors for domain: news_article...
✨ Success! Generated 1159 clean searchable vectors across all documents.


In [33]:
import re

import numpy as np
import spacy
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer


def get_nlp_model():
    """
    Loads the spaCy English model.

    Returns:
        spacy.Language: The loaded spaCy pipeline.
    """
    return spacy.load("en_core_web_sm")


def get_embedding_model(model_name: str = "all-MiniLM-L6-v2") -> SentenceTransformer:
    """
    Loads the sentence-transformer embedding model.

    Args:
        model_name (str): HuggingFace sentence-transformers model identifier.

    Returns:
        SentenceTransformer: The loaded embedding model.
    """
    return SentenceTransformer(model_name)


def clean_query(text: str) -> str:
    """
    Cleans and lemmatizes the user query the same way the corpus chunks
    were cleaned: lowercase, stopword removal, lemmatization.

    Args:
        text (str): Raw user query.

    Returns:
        str: Cleaned, lemmatized query string.
    """
    doc = get_nlp_model()(text.lower())
    return " ".join(tok.lemma_ for tok in doc if not tok.is_stop and tok.is_alpha)


def clean_snippet(text: str, max_chars: int = 50) -> str:
    """
    Flattens whitespace/newlines and truncates text for clean table printing.

    Args:
        text (str): Raw chunk text.
        max_chars (int): Number of characters to keep before truncating.

    Returns:
        str: A single-line, truncated snippet.
    """
    flattened = re.sub(r"\s+", " ", text).strip()
    return flattened[:max_chars] + "..."


def run_retrieval_comparison(processed_chunks: list) -> dict:
    """
    Asks the user for a query, then retrieves the top-1 most relevant
    chunk using three methods — BoW, TF-IDF, and sentence-transformer
    embeddings (all-MiniLM-L6-v2). Prints a comparison table of each
    method's top result and score, and returns the chunk chosen by
    the embedding method (highest-probability match) as the final
    context for the next phase (LLM generation).

    Args:
        processed_chunks (list): List of dicts, each with keys
            "source", "raw_text", and "cleaned_text".

    Returns:
        dict: The chunk dict selected by dense embeddings.
    """
    if not processed_chunks:
        raise ValueError("processed_chunks is empty — nothing to search over.")

    user_query = input("🔎 Enter your query: ").strip()
    print(f"\n🔍 Active Target Search Query: '{user_query}'")

    cleaned_corpus = [chunk["cleaned_text"] for chunk in processed_chunks]
    raw_corpus = [chunk["raw_text"] for chunk in processed_chunks]
    cleaned_query = clean_query(user_query)
    print(f"🧹 Cleaned Query for Keyword Matching: '{cleaned_query}'\n")

    # ---- METHOD 1: Bag of Words (BoW) ----
    bow_vectorizer = CountVectorizer()
    bow_matrix = bow_vectorizer.fit_transform(cleaned_corpus)
    bow_query_vector = bow_vectorizer.transform([cleaned_query])
    bow_scores = cosine_similarity(bow_query_vector, bow_matrix).flatten()
    best_bow_idx = int(np.argmax(bow_scores))

    # ---- METHOD 2: TF-IDF ----
    tfidf_vectorizer = TfidfVectorizer()
    tfidf_matrix = tfidf_vectorizer.fit_transform(cleaned_corpus)
    tfidf_query_vector = tfidf_vectorizer.transform([cleaned_query])
    tfidf_scores = cosine_similarity(tfidf_query_vector, tfidf_matrix).flatten()
    best_tfidf_idx = int(np.argmax(tfidf_scores))

    # ---- METHOD 3: Sentence-Transformer Embeddings ----
    embedding_model = get_embedding_model()
    chunk_embeddings = embedding_model.encode(raw_corpus, convert_to_numpy=True)
    query_embedding = embedding_model.encode([user_query], convert_to_numpy=True)
    embedding_scores = cosine_similarity(query_embedding, chunk_embeddings).flatten()
    best_embedding_idx = int(np.argmax(embedding_scores))

    # ---- Print comparison table ----
    print("📊 RETRIEVAL METHOD COMPARISON TABLE")
    print(f"{'Method':<22} | {'Score':<6} | {'Source':<18} | {'Top Snippet'}")
    print("-" * 100)
    print(f"{'Bag of Words (BoW)':<22} | {bow_scores[best_bow_idx]:.4f} | {processed_chunks[best_bow_idx]['source']:<18} | {clean_snippet(processed_chunks[best_bow_idx]['raw_text'])}")
    print(f"{'TF-IDF':<22} | {tfidf_scores[best_tfidf_idx]:.4f} | {processed_chunks[best_tfidf_idx]['source']:<18} | {clean_snippet(processed_chunks[best_tfidf_idx]['raw_text'])}")
    print(f"{'Dense Embeddings':<22} | {embedding_scores[best_embedding_idx]:.4f} | {processed_chunks[best_embedding_idx]['source']:<18} | {clean_snippet(processed_chunks[best_embedding_idx]['raw_text'])}")
    print("-" * 100)

    final_context_chunk = processed_chunks[best_embedding_idx]
    print(f"🎯 Highest-probability match: '{final_context_chunk['source']}' (via Dense Embeddings)")
    print("🚀 Context isolated for the LLM generation phase.\n")

    return final_context_chunk

In [34]:
final_context = run_retrieval_comparison(chunked_corpus)

🔎 Enter your query: What are the recent advancements in artificial intelligence and how are they being applied in real-world systems?

🔍 Active Target Search Query: 'What are the recent advancements in artificial intelligence and how are they being applied in real-world systems?'
🧹 Cleaned Query for Keyword Matching: 'recent advancement artificial intelligence apply real world system'



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

📊 RETRIEVAL METHOD COMPARISON TABLE
Method                 | Score  | Source             | Top Snippet
----------------------------------------------------------------------------------------------------
Bag of Words (BoW)     | 0.4500 | research_paper     | promotes the production of new artificial intellig...
TF-IDF                 | 0.3576 | research_paper     | promotes the production of new artificial intellig...
Dense Embeddings       | 0.6005 | research_paper     | . It is becoming essential for today's time becaus...
----------------------------------------------------------------------------------------------------
🎯 Highest-probability match: 'research_paper' (via Dense Embeddings)
🚀 Context isolated for the LLM generation phase.



In [35]:
import json
import os
import time

from google import genai
from google.genai import types
from google.genai.errors import APIError
from google.colab import userdata

CACHE_PATH = "gemini_response_cache.json"
MODEL_NAME = "gemini-2.5-flash"  # cheaper/free-tier-friendly than gemini-2.5-pro


def get_gemini_client(api_key: str) -> genai.Client:
    """
    Creates a Gemini API client.

    Args:
        api_key (str): Your Gemini API key (e.g. from Colab Secrets).

    Returns:
        genai.Client: An authenticated Gemini client.
    """
    return genai.Client(api_key=api_key)


def build_prompt(user_query: str, context_chunk: dict) -> str:
    """
    Builds a structured prompt using Role + Context + Few-Shot Examples +
    Output Format (JSON), for grounded question-answering over a retrieved chunk.

    Args:
        user_query (str): The user's original question.
        context_chunk (dict): The winning retrieved chunk, with keys
            "source" and "raw_text" (or "text").

    Returns:
        str: The fully assembled prompt string.
    """
    context_text = context_chunk.get("raw_text") or context_chunk.get("text", "")
    source = context_chunk.get("source", "unknown")

    prompt = f"""
<role>
You are a precise research assistant. You answer questions strictly using
the provided context. You never invent facts that are not supported by it.
</role>

<context source="research_paper.pdf">
Artificial intelligence research has advanced significantly in recent years,
particularly in the areas of natural language processing and computer vision.
These advancements have enabled the deployment of AI systems in real-world
applications such as healthcare diagnostics, autonomous vehicles, and
customer service automation.
</context>

<few_shot_examples>
Example 1
Question: What does the context say about renewable energy adoption?
Answer: {{"answer": "The context states that renewable energy adoption increased due to falling solar panel costs.", "supported": true, "source": "example_doc.pdf"}}

Example 2
Question: What is the capital of France?
Answer: {{"answer": "The context does not contain information to answer this question.", "supported": false, "source": "example_doc.pdf"}}
</few_shot_examples>

<output_format>
Respond ONLY with valid JSON, no extra text, no markdown fences, matching exactly:
{{"answer": "<your answer>", "supported": <true or false>, "source": "research_paper.pdf"}}
</output_format>

<question>
What are the recent advancements in artificial intelligence and how are they being applied in real-world systems?
</question>
""".strip()

    return prompt


def _load_cache() -> dict:
    """Loads the local response cache from disk, or returns an empty dict."""
    if os.path.exists(CACHE_PATH):
        with open(CACHE_PATH, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}


def _save_cache(cache: dict) -> None:
    """Writes the local response cache to disk."""
    with open(CACHE_PATH, "w", encoding="utf-8") as f:
        json.dump(cache, f, indent=2)


def call_gemini(
    client: genai.Client,
    prompt: str,
    temperature: float,
    cache_key: str,
    max_retries: int = 3,
) -> str:
    """
    Calls Gemini with the given prompt and temperature. Uses a local cache
    to avoid repeat API calls for the same input while debugging, and
    retries with exponential backoff on transient rate-limit errors.

    Args:
        client (genai.Client): An authenticated Gemini client.
        prompt (str): The full structured prompt to send.
        temperature (float): Sampling temperature (e.g. 0.1 or 0.9).
        cache_key (str): A unique key identifying this exact call
            (e.g. f"{query}|{chunk_id}|{temperature}").
        max_retries (int): Number of retry attempts on rate-limit errors.

    Returns:
        str: The raw text response from Gemini (expected to be JSON).
    """
    cache = _load_cache()
    if cache_key in cache:
        print(f"💾 Using cached response for temperature={temperature} (no API call made).")
        return cache[cache_key]

    for attempt in range(1, max_retries + 1):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config=types.GenerateContentConfig(temperature=temperature),
            )
            result_text = response.text
            cache[cache_key] = result_text
            _save_cache(cache)
            return result_text

        except APIError as e:
            error_str = str(e)

            if "RESOURCE_EXHAUSTED" in error_str or "429" in error_str:
                if "PerDay" in error_str or "daily" in error_str.lower():
                    raise RuntimeError(
                        "Gemini free-tier DAILY quota is exhausted. "
                        "This will reset after 24 hours (UTC). "
                        "No amount of retrying will fix this right now — "
                        "wait for the reset, or use a different API key/project."
                    ) from e

                # Likely a per-minute rate limit — worth retrying with backoff
                wait_time = 2 ** attempt
                print(f"⏳ Rate limited (attempt {attempt}/{max_retries}). Waiting {wait_time}s before retry...")
                time.sleep(wait_time)
                continue

            # Some other API error — don't silently retry, surface it
            raise

    raise RuntimeError(
        f"Failed after {max_retries} retries due to persistent rate limiting. "
        "Try again in a few minutes."
    )


def generate_dual_temperature_answers(
    client: genai.Client,
    user_query: str,
    context_chunk: dict,
) -> dict:
    """
    Generates two Gemini answers for the same query+context — one at low
    temperature (0.1, deterministic/focused) and one at high temperature
    (0.9, more varied/creative) — and packages the comparison.

    Args:
        client (genai.Client): An authenticated Gemini client.
        user_query (str): The user's original question.
        context_chunk (dict): The winning retrieved chunk from Stage 2.

    Returns:
        dict: {"low_temp_answer": ..., "high_temp_answer": ...}
    """
    prompt = build_prompt(user_query, context_chunk)
    chunk_id = context_chunk.get("chunk_id", context_chunk.get("source", "chunk"))

    print("🤖 Generating answer at temperature=0.1 ...")
    low_temp_key = f"{user_query}|{chunk_id}|0.1"
    low_temp_answer = call_gemini(client, prompt, temperature=0.1, cache_key=low_temp_key)

    # Small pause to be gentle on per-minute rate limits between the two calls
    time.sleep(2)

    print("🤖 Generating answer at temperature=0.9 ...")
    high_temp_key = f"{user_query}|{chunk_id}|0.9"
    high_temp_answer = call_gemini(client, prompt, temperature=0.9, cache_key=high_temp_key)

    print("\n📋 COMPARISON")
    print(f"Temp 0.1 (focused):  {low_temp_answer}")
    print(f"Temp 0.9 (varied):   {high_temp_answer}")

    return {
        "low_temp_answer": low_temp_answer,
        "high_temp_answer": high_temp_answer,
    }

# from stage3_gemini import get_gemini_client, generate_dual_temperature_answers

GEMINI_API_KEY = userdata.get('Gemini_API_Key')  # store as Colab Secret
client = get_gemini_client(GEMINI_API_KEY)

# final_context comes from your Stage 2 retrieval function
answers = generate_dual_temperature_answers(client, "What are recent advancements in artificial intelligence?", final_context)


🤖 Generating answer at temperature=0.1 ...
💾 Using cached response for temperature=0.1 (no API call made).
🤖 Generating answer at temperature=0.9 ...
💾 Using cached response for temperature=0.9 (no API call made).

📋 COMPARISON
Temp 0.1 (focused):  {"answer": "Artificial intelligence research has advanced significantly in recent years, particularly in the areas of natural language processing and computer vision. These advancements are being applied in real-world systems such as healthcare diagnostics, autonomous vehicles, and customer service automation.", "supported": true, "source": "research_paper.pdf"}
Temp 0.9 (varied):   {"answer": "Artificial intelligence research has advanced significantly in recent years, particularly in the areas of natural language processing and computer vision. These advancements are being applied in real-world systems such as healthcare diagnostics, autonomous vehicles, and customer service automation.", "supported": true, "source": "research_paper.pdf"}
